### Determine current BTC market

In [1]:
from time import time
from datetime import datetime
from zoneinfo import ZoneInfo
from datetime import timezone

WINDOW_SECS = 300  # 5-min window

def current_window_start() -> int:
    """
    Unix timestamp of the current 5-min window's start.
    """
    return (int(time()) // WINDOW_SECS) * WINDOW_SECS


def to_EST(ts: int) -> str:
    dt = datetime.fromtimestamp(ts, tz=timezone.utc).astimezone(ZoneInfo("America/New_York"))
    return dt.strftime("%Y-%m-%d %I:%M:%S %p EST")


def current_window_slug(ts) -> str:
    """
    Current 5-min window slug.
    """
    return f"btc-updown-5m-{ts}"


start = current_window_start()
start_EST = to_EST(start)
slug = current_window_slug(start)
print([start, start_EST, slug])

[1774384800, '2026-03-24 04:40:00 PM EST', 'btc-updown-5m-1774384800']


### Init "clob" Client

In [ ]:

from py_clob_client.client import ClobClient
from app_config import load_config

config = load_config()

host = "https://clob.polymarket.com"
chain_id = 137  # Polygon mainnet

# Derive API credentials (L1 → L2 auth)
temp_client = ClobClient(host, key=config.polymarket.private_key, chain_id=chain_id)
api_creds = temp_client.create_or_derive_api_creds()

# Initialize trading client
client = ClobClient(
    host,
    key=config.polymarket.private_key,
    chain_id=chain_id,
    creds=api_creds,
    # I'm going with the proxy wallet through polymarket to avoid paying gas fees. This seemed like the best
    # one to use from: https://docs.polymarket.com/trading/overview#signature-types.
    signature_type=1,
    funder=config.polymarket.wallet_address,
)
client

### Get active BTC up / down Market

In [69]:
from requests import get as GET
from json import loads
from requests.exceptions import HTTPError

class MarketNotFound(Exception):
    ...

def get_market_by_slug(slug: str) -> dict:
    """Fetch a single BTC 5-min market by its exact slug."""
    response = GET(f"https://gamma-api.polymarket.com/events", params={"slug": slug}, timeout=10)
    try:
        response.raise_for_status()
        data = response.json()
        if data is None or not isinstance(data, list) or len(data) < 1:
            raise MarketNotFound(f"No market data found for slug: {slug}.")
        return data[0]
    except HTTPError as e:
        raise MarketNotFound from e
    

market = get_market_by_slug(slug)
if len(market["markets"]) != 1:
    raise AssertionError("Expected BTC Up/Down market response to contain exactly 1 market!")
outcomes = loads(market["markets"][0]["outcomes"])
tids = loads(market["markets"][0]["clobTokenIds"])
market_clobs = dict(zip([x.lower() for x in outcomes], tids))
market

{'id': '279967',
 'ticker': 'btc-updown-5m-1773848100',
 'slug': 'btc-updown-5m-1773848100',
 'title': 'Bitcoin Up or Down - March 18, 11:35AM-11:40AM ET',
 'description': 'This market will resolve to "Up" if the Bitcoin price at the end of the time range specified in the title is greater than or equal to the price at the beginning of that range. Otherwise, it will resolve to "Down".\nThe resolution source for this market is information from Chainlink, specifically the BTC/USD data stream available at https://data.chain.link/streams/btc-usd.\nPlease note that this market is about the price according to Chainlink data stream BTC/USD, not according to other sources or spot markets.',
 'resolutionSource': 'https://data.chain.link/streams/btc-usd',
 'startDate': '2026-03-17T15:43:39.56641Z',
 'creationDate': '2026-03-17T15:43:39.566406Z',
 'endDate': '2026-03-18T15:40:00Z',
 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png',
 'icon': 'https://polymarket-uploa

In [17]:
up_book = client.get_order_book(market_clobs["down"])
up_book.asks

NameError: name 'market_clobs' is not defined

In [2]:
from market_info import get_current_market_info
from clob_client import get_client
from py_clob_client.clob_types import BalanceAllowanceParams, AssetType, MarketOrderArgs, OrderType
from py_clob_client.order_builder.constants import BUY


client = get_client()

balance_allowance_response = client.get_balance_allowance(BalanceAllowanceParams(asset_type=AssetType.COLLATERAL))
balance = float(balance_allowance_response["balance"]) / 1e6
balance

100.045163

In [23]:
info = get_current_market_info()


from py_clob_client.clob_types import MarketOrderArgs
from py_clob_client.clob_types import OrderType
from py_clob_client.order_builder.constants import BUY

order = client.create_market_order(
    MarketOrderArgs(
        token_id=info.up_clob_id,
        amount=1,
        side=BUY,
        price=0.95, # slippage ceiling — won't pay more than this
    )
)

response = client.post_order(order, OrderType.FOK)
response
# response["status"] == "matched" → filled
# response["status"] == "cancelled" → no liquidity at that price


{'errorMsg': '',
 'orderID': '0xd2cdc0855a135c38b317d9064e2b94ca441b1e97c3a51a09fc8d49f741036b01',
 'takingAmount': '1.234563',
 'makingAmount': '0.999996',
 'status': 'matched',
 'transactionsHashes': ['0x163c1f11dacef392b8618f4876a0410666e25e0065273bd162e5628f3f46c52a'],
 'success': True}

In [10]:
from py_clob_client.clob_types import TradeParams

trades = client.get_trades(TradeParams())
trades  # list of filled trades with market, side, price, size, status

[{'id': '5489cf43-1bab-44f2-b3ff-35f309e7443b',
  'taker_order_id': '0xd2cdc0855a135c38b317d9064e2b94ca441b1e97c3a51a09fc8d49f741036b01',
  'market': '0xe498095a299da58676805699f83de4d559217647bca5ca7d1a68314ad7bbf22e',
  'asset_id': '102427150759342266591321547577410221787925585071702027090492867812129102220303',
  'side': 'BUY',
  'size': '1.234563',
  'fee_rate_bps': '1000',
  'price': '0.81',
  'status': 'CONFIRMED',
  'match_time': '1774386177',
  'last_update': '1774386211',
  'outcome': 'Up',
  'bucket_index': 0,
  'owner': '29f82706-4101-9f7d-71f5-dccc3e4eaf28',
  'maker_address': '0x268a223CB2B2f9742904319613fEc3c36c3749e8',
  'transaction_hash': '0x163c1f11dacef392b8618f4876a0410666e25e0065273bd162e5628f3f46c52a',
  'maker_orders': [{'order_id': '0xa80cb53f7b1496d13536fb5e282f07254e81bb46ef07417c97204965d1c1b318',
    'owner': '051bb939-eb54-b3ac-7b72-7f5b63aa12b7',
    'maker_address': '0xB27BC932bf8110D8F78e55da7d5f0497A18B5b82',
    'matched_amount': '1.234563',
    'price